# Lab 05 guiado · Aprendizaje no supervisado

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 5, ruta guiada*

Este cuaderno se abre directo desde GitHub. Lo que escribas se pierde al cerrar la pestaña. Si quieres conservarlo, descárgalo con Archivo, Descargar, Descargar .ipynb.

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#31708f"></td><td bgcolor="#d9edf7"><font color="#0b3d62" size="3"><b>Qué vamos a responder</b><br>1. Si solo miramos cómo generan durante el día, qué centrales se parecen entre sí.<br>2. Cómo se juntan en grupos sin que nadie diga de antemano cuáles son.<br>3. Qué centrales no se parecen a nada, y por qué eso también es un hallazgo.</font></td></tr></table>

El lab tiene dos secciones. **En clase** tiene tres partes y se hace en clase, una detrás de la otra. **Para estudiar** tiene tres estudios para después, con la guía, y se corre solo después de la sección En clase.

---
<br>

# En clase

Tres partes, una detrás de la otra. Córrelas en orden.

---
<br>

## Parte 1 de 3 · Comparar por la forma, no por el tamaño

**Qué vamos a responder.** Cómo se describe cada central con 24 números, su día promedio, y por qué hay que dividir por la potencia antes de comparar.

![Dos centrales en megawatts y divididas por su potencia](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_parte1_forma.png)

**Qué hacer.** Corre la celda. En la tabla cada fila es una central, una por tecnología, y cada columna una hora del día.

In [ ]:
# Parte 1
DATOS = "https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/datos/"
import pandas as pd

# La generación de cada hora de 2024 y la ficha de las 20 centrales, como en el Lab 02.
gen = pd.read_csv(DATOS + "generacion.csv", parse_dates=["fecha"])
centrales = pd.read_csv(DATOS + "centrales.csv")

# Una fila por central con 24 columnas, el promedio de cada hora. Es su día promedio.
perfil = gen.pivot_table(index="central", columns="hora", values="mwh", aggfunc="mean")

# Dividir cada fila por la potencia de esa central. Ahora cada número es la parte
# de su potencia que usa en esa hora, entre 0 y 1. Se compara la forma, no el tamaño.
potencia = centrales.set_index("central")["potencia_mw"]
factor = perfil.div(potencia, axis=0)

print("Centrales y horas", factor.shape)
print("Antes, de", round(perfil.values.min(), 1), "a", round(perfil.values.max(), 1), "MWh")
print("Después, de", round(factor.values.min(), 3), "a", round(factor.values.max(), 3))

# Una central de cada tecnología, de las 9 a las 14 horas.
tecnologia = centrales.set_index("central")["tecnologia"]
muestra = factor.index.to_series().groupby(tecnologia[factor.index]).first()
factor.loc[muestra].round(2).iloc[:, 9:15]

---
<br>

## Parte 2 de 3 · K-Means, juntar lo que se parece

**Qué vamos a responder.** Si le damos las 20 centrales a K-Means y le pedimos 6 grupos, sin decirle la tecnología de ninguna, junta las que son de la misma tecnología.

![Los seis grupos de K-Means, con el color de la tecnología real](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_parte2_kmeans6.png)

**Qué hacer.** Corre la celda. La tabla cruza el grupo que eligió K-Means con la tecnología real, que el algoritmo nunca vio.

In [ ]:
# Parte 2
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# X es la tabla factor de la Parte 1 de 3, pasada por StandardScaler. Eso deja cada hora
# centrada en cero y con el mismo ancho, para que ninguna hora pese más que otra.
X = StandardScaler().fit_transform(factor)

# K-Means pide cuántos grupos. Reparte las centrales y ajusta hasta que nada cambia.
km6 = KMeans(n_clusters=6, random_state=2026, n_init=10).fit(X)

# La tecnología real, que K-Means no vio. Cada fila una tecnología, cada columna un grupo.
pd.crosstab(tecnologia[factor.index], km6.labels_)

---
<br>

### Tu turno · ¿Cuántos grupos pedir?

**Qué vamos a responder.** Cómo se elige el número de grupos cuando nadie lo sabe de antemano. Se prueba con varios y se mira cuánto desorden queda dentro de los montones.

![La inercia baja al agregar grupos](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_parte2_codo.png)

**Qué hacer.** Corre la celda. Después cambia el 6 por 4 en la lista, y vuelve a correrla. Mira qué tecnologías quedan juntas.

In [ ]:
# Parte 2, tu turno
grupos = 6 #@param [4, 5, 6, 7]

# Para cada cantidad de grupos, cuánto desorden queda dentro de los montones.
# El programa le llama inercia. Siempre baja al agregar grupos.
for k in range(1, 9):
    inercia = KMeans(n_clusters=k, random_state=2026, n_init=10).fit(X).inertia_
    print(k, "grupos, inercia", round(inercia, 1))

# Y la tabla con la cantidad de grupos elegida.
km = KMeans(n_clusters=int(grupos), random_state=2026, n_init=10).fit(X)
print()
print("Con", grupos, "grupos")
pd.crosstab(tecnologia[factor.index], km.labels_)

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>Cuántos grupos pedir es una decisión</b><br>Con 4 grupos, carbón, gas e hidro quedan juntas, porque las tres trabajan parejo todo el día.<br>Con 6 grupos cada tecnología queda en su propio grupo, sin que nadie se lo dijera.<br>La inercia cae fuerte hasta 3 grupos y desde 5 ya casi es cero. Se busca hasta dónde vale la pena agregar grupos.<br>Si se agrupara en megawatts sin dividir por la potencia, K-Means juntaría grandes con grandes y las tecnologías saldrían mezcladas.</font></td></tr></table>

---
<br>

## Parte 3 de 3 · DBSCAN, y lo que no cabe en ningún grupo

**Qué vamos a responder.** Qué pasa cuando en vez de una fila por central hay una fila por central y día, y se usa un algoritmo que no pide número de grupos sino un radio.

![Los días que DBSCAN dejó fuera de todo grupo](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_parte3_ruido.png)

**Qué hacer.** Corre la celda. La columna menos uno es el ruido, los días que no cupieron en ningún grupo.

In [ ]:
# Parte 3
from sklearn.cluster import DBSCAN

# Ahora una fila por central y día. 20 centrales por 366 días son 7.320 filas.
perfil_dia = gen.pivot_table(index=["central", "fecha"], columns="hora", values="mwh", aggfunc="mean")
potencia_dia = perfil_dia.index.get_level_values("central").map(potencia)
factor_dia = perfil_dia.div(potencia_dia, axis=0)
X_dia = StandardScaler().fit_transform(factor_dia)
print("Filas y horas", X_dia.shape)

# DBSCAN no pide grupos. Pide un radio, eps, qué tan cerca tienen que estar dos días
# para ser vecinos, y cuántos vecinos hacen falta para formar un grupo, min_samples.
# Los días sin suficientes vecinos quedan con la etiqueta menos uno, el ruido.
db = DBSCAN(eps=1.5, min_samples=5).fit(X_dia)

tecnologia_dia = perfil_dia.index.get_level_values("central").map(tecnologia)
pd.crosstab(tecnologia_dia, db.labels_)

---
<br>

### Tu turno · Mueve el radio

**Qué vamos a responder.** Cómo cambia lo que encuentra DBSCAN cuando se cambia el radio.

![Cinco días de una solar y cinco de una eólica](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_parte3_solar_eolica.png)

**Qué hacer.** Corre la celda con el radio en 1,5. Después prueba 3,0 y 3,5 y mira qué pasa con el ruido.

In [ ]:
# Parte 3, tu turno
radio = 1.5 #@param [1.5, 2.0, 3.0, 3.5]

prueba = DBSCAN(eps=float(radio), min_samples=5).fit(X_dia)
cuantos = len(set(prueba.labels_)) - (1 if -1 in prueba.labels_ else 0)
ruido = int((prueba.labels_ == -1).sum())

print("Radio", radio)
print("Grupos", cuantos)
print("Días fuera de todo grupo, ruido", ruido, "de", len(X_dia))
print("Días eólicos en la tabla", int((tecnologia_dia == "eolica").sum()))

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>El ruido no siempre es un error</b><br>Con radio 1,5 quedan 1.464 días fuera de todo grupo. Son exactamente las 4 eólicas por 366 días.<br>Ningún día de una eólica se parece lo suficiente a otro como para formar un grupo, porque su producción la manda el viento.<br>Con radio 3,5 ya no queda ruido, pero porque el radio es tan grande que mete a todos en algún grupo y se pierde el hallazgo.<br>El ruido se mira antes de descartarlo. Aquí es la respuesta a la tercera pregunta del lab.</font></td></tr></table>

---
<br>

# Para estudiar

Estos estudios se hacen después de clase, con la guía en PDF. Antes de empezar, corre las tres partes de En clase, porque usan sus tablas.

---
<br>

## Estudio 1 de 3 · ¿Cómo se dibujan 24 números en un plano?

**Qué vamos a responder.** Cómo PCA resume los 24 números de cada central en 2, para poder dibujar las 20 centrales en un plano y ver quién queda cerca de quién.

![Las veinte centrales en dos dimensiones](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_estudio1_pca.png)

**Qué hacer.** Corre la celda. Las centrales de una misma tecnología caen casi en el mismo punto.

In [ ]:
# Estudio 1
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# PCA busca las dos direcciones donde los datos más se estiran y se queda con esas.
pca = PCA(n_components=2).fit(X)
Z = pca.transform(X)
print("Cuánto de la variación conservan las dos componentes", pca.explained_variance_ratio_.round(3))

mapa = pd.DataFrame(Z, columns=["pc1", "pc2"], index=factor.index)
mapa["tecnologia"] = tecnologia[mapa.index].values

fig, ax = plt.subplots(figsize=(7, 5))
for t, g in mapa.groupby("tecnologia"):
    ax.scatter(g["pc1"], g["pc2"], label=t, s=45, alpha=0.75)
ax.set_title("Las veinte centrales en dos dimensiones")
ax.set_xlabel("componente 1")
ax.set_ylabel("componente 2")
ax.legend()
plt.show()

---
<br>

## Estudio 2 de 3 · ¿Los grupos quedaron bien?

**Qué vamos a responder.** Cómo se mide si los grupos quedaron bien separados sin tener la respuesta correcta. La silueta va de menos uno a uno.

![La silueta para cada cantidad de grupos](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_estudio2_silueta.png)

**Qué hacer.** Corre la celda. Después compara la silueta con la inercia de la Parte 2 de 3.

In [ ]:
# Estudio 2
from sklearn.metrics import silhouette_score

# La silueta mide, para cada punto, qué tan cerca está de su grupo comparado con
# el grupo vecino. Cerca de 1 es un grupo bien separado. No necesita la respuesta.
for k in range(2, 8):
    etiquetas = KMeans(n_clusters=k, random_state=2026, n_init=10).fit_predict(X)
    print(k, "grupos, silueta", round(silhouette_score(X, etiquetas), 3))

---
<br>

## Estudio 3 de 3 · Temas en el texto libre

**Qué vamos a responder.** Cómo LDA encuentra temas en las observaciones de la bitácora, sin leerlas, y si esos temas se parecen a los tipos de evento.

![Los temas que encontró LDA contra el tipo de evento](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo05/img/lab05_estudio3_lda.png)

**Qué hacer.** Corre la celda. Después cambia el 6 por 3 y mira si los temas siguen separando los tipos.

In [ ]:
# Estudio 3
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

temas = 6 #@param [3, 6]

# La bitácora del Lab 04, con la columna de texto que no habíamos usado.
mant = pd.read_csv(DATOS + "mantenimiento.csv")

# Cada observación pasa a ser un conteo de palabras. Las palabras que están en todas
# partes no distinguen nada, y se sacan.
VACIAS = ["el", "la", "los", "las", "un", "una", "de", "del", "en", "y", "se", "que", "por", "con",
          "sobre", "tras", "su", "sus", "al", "lo", "es", "durante", "ronda", "turno", "registra",
          "detecta", "reporta", "evento", "operador", "revisa"]
# max_df saca las palabras que están en más del 90 % de los textos, min_df las que aparecen en menos de 3.
vectorizador = CountVectorizer(stop_words=VACIAS, max_df=0.9, min_df=3)
matriz = vectorizador.fit_transform(mant["observacion"])

# LDA busca temas. No sabe que existen seis tipos de evento.
lda = LatentDirichletAllocation(n_components=int(temas), random_state=2026).fit(matriz)
palabras = vectorizador.get_feature_names_out()
for i, tema in enumerate(lda.components_):
    print(f"tema {i}:", ", ".join(palabras[j] for j in tema.argsort()[-6:][::-1]))

# A cada texto su tema dominante, cruzado con el tipo de evento, que LDA nunca vio.
dominante = lda.transform(matriz).argmax(axis=1)
pd.crosstab(mant["tipo_evento"], dominante)

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>Agrupar sin respuesta correcta</b><br>K-Means, DBSCAN y LDA encontraron grupos sin que nadie les dijera cuáles eran.<br>En el curso podemos comparar contra la tecnología o el tipo de evento, porque los datos son inventados y los conocemos.<br>En el trabajo real esa comparación no existe. Por eso se usan el codo, la silueta y el criterio de quien conoce el tema.</font></td></tr></table>

---
<br>

## Lo que lograste hoy

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#28a745"></td><td bgcolor="#d8f0df"><font color="#14532d" size="3">• Cada central se puede describir con 24 números, su día promedio, y se compara por la forma dividiendo por la potencia.<br>• K-Means junta lo que se parece. Tú le dices cuántos grupos, él reparte hasta que queda estable.<br>• Con 6 grupos y comparando por la forma, recuperó las 6 tecnologías sin conocerlas.<br>• Cuántos grupos pedir se decide mirando hasta dónde la inercia deja de bajar.<br>• DBSCAN no pide grupos, pide un radio, y lo que no cabe en ninguno lo llama ruido.<br>• El ruido se mira antes de descartarlo. Aquí fueron los 1.464 días de las eólicas.</font></td></tr></table>

## Para seguir practicando

Estos ejercicios son para quien quiera seguir después de clase. Aquí sí se escribe código.
Antes de empezar, corre todo el cuaderno con Entorno de ejecución, Ejecutar todo.
Escribe tu respuesta en la celda vacía de cada ejercicio y abre la solución solo después de intentarlo.

### Ejercicio 1 · Los centros de cada grupo

El centro de un grupo es el promedio de sus centrales. Calcula el promedio de factor por grupo de km6 y mira las horas 0, 3, 12 y 21. Di cuál es el grupo de las solares y cuál el del diésel. Pista, factor.groupby(km6.labels_).mean() da una fila por grupo.

In [ ]:
# Ejercicio 1


<details>
<summary>Solución · Ejercicio 1</summary>

```python
centros = factor.groupby(km6.labels_).mean()
centros[[0, 3, 12, 21]].round(2)
```

El grupo de las solares es el único con cero a las 0, 3 y 21 horas y un valor alto a las 12. El del diésel es el único con cero a las 0, 3 y 12 y un valor alto a las 21. Los otros cuatro son parejos a toda hora, cada uno en su nivel, el de las eólicas más bajo.

</details>

### Ejercicio 2 · K-Means sin dividir por la potencia

Repite la Parte 2 de 3 usando perfil en vez de factor, es decir, en megawatts tal como vienen. Cruza los 6 grupos con la tecnología y cuenta cuántos grupos mezclan tecnologías distintas.

In [ ]:
# Ejercicio 2


<details>
<summary>Solución · Ejercicio 2</summary>

```python
X_crudo = StandardScaler().fit_transform(perfil)
km_crudo = KMeans(n_clusters=6, random_state=2026, n_init=10).fit(X_crudo)
cruce = pd.crosstab(tecnologia[perfil.index], km_crudo.labels_)
print(cruce)
print("Grupos que mezclan tecnologías", int((cruce.gt(0).sum() > 1).sum()))
```

Las 5 hidro quedan repartidas en varios grupos y hay grupos con hidro, gas y carbón mezclados. Sin dividir por la potencia, K-Means agrupa por tamaño.

</details>

### Ejercicio 3 · La silueta de DBSCAN

Calcula la silueta de la solución de DBSCAN con radio 3,0, sacando antes los puntos de ruido, y compárala con la de K-Means con 6 grupos en X. Pista, la silueta no acepta la etiqueta menos uno.

In [ ]:
# Ejercicio 3


<details>
<summary>Solución · Ejercicio 3</summary>

```python
db3 = DBSCAN(eps=3.0, min_samples=5).fit(X_dia)
sin_ruido = db3.labels_ != -1
print("Puntos que quedan", int(sin_ruido.sum()), "de", len(X_dia))
print("Silueta DBSCAN", round(silhouette_score(X_dia[sin_ruido], db3.labels_[sin_ruido]), 3))
print("Silueta K-Means 6", round(silhouette_score(X, km6.labels_), 3))
```

Son dos tablas distintas, una fila por central contra una fila por central y día, así que las siluetas no se comparan directo. Lo que importa es que DBSCAN deja 5 puntos fuera con radio 3,0 y K-Means no deja ninguno.

</details>